# Table 2 from the raw outputs

Table 2 of the paper is a digest of the registered rules its argument rests on. This notebook recomputes each row
from the raw records in this repository (one line per replay or per episode, as the runs wrote them) and prints the
result beside the value that the registered analysis stored. It is committed with its outputs: nothing has to be run
to read it.

**How to read a row.** A text cell gives the rule as registered and the file under `prereg/` that holds it. The code
cell under it goes from the records to a rate per condition, to the difference, to its interval. It prints the counts
per condition, then one line:

`recomputed: ... | stored by the registered analysis: ... | equal: yes/no`

The stored value is read from the report that the analysis script printed, kept under `analysis/results/`
(`analysis/reproduce.py` runs the analyses again and compares their output with these files). No result is typed
here. The last cell prints the whole table again.

To run it again: `py -I -X utf8 notebooks/run_notebook.py` from the repository root, which needs no Jupyter, or run
all cells in Jupyter. `notebooks/README.md` says what data that takes.

**Words.** A *paused step* is a recorded run of the starter agent, cut after a tool result that shows file content,
together with one function the agent has just seen. A *replay* loads that history into the harness's own agent, adds
one message, and records the agent's next call. *Conditions* differ only in that message: *no list*; *bare names*
(the function's graph neighbors as the tool prints them); *file only*; *file and lines*; *addresses* (file, line
range and first definition line); a *false entry*; a *rewired list*. *Fresh episodes* and *whole runs* start from the
task prompt, with the harness *as shipped*, *addressed* (every entity in a graph tool's answer also carries its file
and line range) or *addressed and repaired* (a misspelled line-bound argument is restored as well). Outcomes: the
next call *reads a neighbor's code* or *opens a neighbor's file*; a read *asks past line 1*; the agent is *shown the
fix* when a tool result shows a line that the reference patch changes.

**What is imported and what is shown.** The scoring of one call or one episode and the bootstrap are imported from
the repository's analysis code, so that the numbers agree to the last digit:

| Imported | From `analysis/` | What it decides |
|---|---|---|
| `any_true`, `touch_map` | `gates_c.py`, on `fork_states.touches` | whether one call reads, views or edits the lines of a listed function, under the harness's caps (150 lines and 10,000 characters per read, 5,000 characters of shell output) |
| `failure`, `first_call_failure` | `gates_c.py`, `analyze_k.py` | whether a replay is an interface failure: an error, no call, a tool that does not exist, malformed arguments |
| `opens_listed_file` | `mangled_sensitivity.py` | whether a call is a `read_file` of a neighbor's file that the agent had not opened |
| `read_stats` | `explore_first_reads.py` | whether such a read asks past line 1 |
| `mangled`, `ranged_reads` | `mangled_sensitivity.py`, `analyze_g.py` | whether a read carries a line bound under a misspelled argument name |
| `score` | `analyze_k.py` | over the calls of one replay or episode: whether a tool result shows a line that the reference patch changes |
| `episode` | `analyze_09.py` | in one fresh episode: whether the call after a graph answer reads an entity that the answer returned |
| `reads_of` | `analyze_11.py` | the file reads of one episode, each with what the call just before showed about that file |
| `load_gold` | `fork_states.py` | which functions and files each task's reference patch changes |
| `boot_ci`, `diff_ci` | `analyze_forks.py` | the bootstrap: 2,000 resamples of clusters, seed 0 |

Everything between a scored record and a printed number is in the cells below: which records are usable, which
paused steps or tasks are complete, the rate per condition, the difference and its interval.

**Intervals** are 95% bootstrap intervals. A cluster is a paused step, or a task for fresh episodes and whole runs.
A difference is in percentage points.

## Setup and inputs

The first cell finds the repository root from the folder this notebook is in, imports the scorers, defines the few
helpers that every row uses, and lists the inputs: whether each is on this machine, whether it is a file of the
repository, and which script makes it.

The task snapshots are competition data and are not in the repository. Five rows do not need them (12 P2, 12 P3,
11 R1, 07 F2, 08 N1). Every other row does, because its scorer applies the harness's character caps to the real
lines of the files, or looks up where a function is. A row whose input is missing says so and is not approximated.

In [1]:
import collections
import json
import re
import subprocess
import sys
from decimal import Decimal
from pathlib import Path

# The repository root, found from the folder this notebook is in. Every path below is relative to it.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'analysis' / 'reproduce.py').is_file())
ANALYSIS, RESULTS = ROOT / 'analysis', ROOT / 'analysis' / 'results'
SNAPSHOTS, INDEX = ROOT / 'data' / 'snapshots', ROOT / 'data' / 'cache' / 'snapshot_files'
sys.path.insert(0, str(ANALYSIS))
if not hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure = lambda **settings: None   # a module imported below calls it; a notebook may lack it

# The repository's scoring of one call or one episode, and its bootstrap: imported, not copied.
from analyze_forks import boot_ci, diff_ci
from gates_c import any_true, failure, touch_map
from mangled_sensitivity import mangled, opens_listed_file
from explore_first_reads import read_stats
from analyze_k import first_call_failure, score
from analyze_g import ranged_reads
from analyze_09 import episode
from analyze_11 import reads_of
from fork_states import load_gold

SEEDS = (1, 2, 3)
TABLE = {}   # row: (recomputed, stored, equal); the last cell prints it


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def read_jsonl(path):
    """The records of a file: one JSON object per line."""
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def step_of(record):
    """The paused step of a record: a task, one of its recorded runs, and a cut of that run."""
    return (record['task'], record['run'], record.get('fork', 0))


def load_paused_steps(path):
    """{paused step: what the scorer needs to know about it, such as the listed neighbors, their files and lines}."""
    return {step_of(state): state for state in read_jsonl(path)}


def usable(records, conditions, seeds=SEEDS, broken=failure, cluster=step_of):
    """{cluster: {(condition, seed): record}}, without the records that `broken` rejects.

    For a replay of one call, `broken` is the frozen interface-failure rule: an error, no call, a tool that does
    not exist, or malformed arguments. Such a replay is counted apart and is never an outcome.
    """
    kept = collections.defaultdict(dict)
    for r in records:
        if r['arm'] in conditions and r['seed'] in seeds and not broken(r):
            kept[cluster(r)][(r['arm'], r['seed'])] = r
    return kept


def complete(records, conditions, seeds=SEEDS):
    """The clusters that are analyzed: every condition has a usable record for every seed."""
    need = {(condition, seed) for condition in conditions for seed in seeds}
    return sorted(k for k, v in records.items() if need <= set(v))


def outcomes(clusters, records, condition, outcome, seeds=SEEDS):
    """{cluster: [0 or 1 for each seed]} for one condition: the table that the bootstrap resamples."""
    return {k: [int(bool(outcome(k, records[k][(condition, seed)]))) for seed in seeds] for k in clusters}


def count(table):
    """'hits of n' over all records of a table {cluster: [0 or 1, ...]}."""
    values = [v for per_cluster in table.values() for v in per_cluster]
    return f'{sum(values)} of {len(values)}'


def rates(tables, note):
    """Print the count and the rate of each condition."""
    parts = []
    for name, table in tables.items():
        values = [v for per_cluster in table.values() for v in per_cluster]
        parts.append(f'{name}: {count(table)} ({100 * sum(values) / len(values):.2f}%)')
    print(' | '.join(parts + [note]))


def points(ci, digits=2):
    """An estimate and its interval in percentage points, rounded as the stored reports round them."""
    return '{:+.{d}f} [{:+.{d}f}, {:+.{d}f}]'.format(*(100 * x for x in ci), d=digits)


def as_printed(ci):
    """An estimate and its interval as the reports of 02, 03 and 04 print them: proportions, four decimals."""
    return '{:.4f} [{:.4f}, {:.4f}]'.format(*ci)


def in_points(proportions):
    """'0.0550 [0.0344, 0.0765]' as '+5.50 [+3.44, +7.65]': the same digits, in percentage points."""
    return re.sub(r'-?\d+\.\d+', lambda number: f'{Decimal(number.group()) * 100:+.2f}', proportions)


def stored(report, pattern, form='{} [{}, {}]'):
    """What a report under analysis/results/ prints for a row: the groups of `pattern`, laid out by `form`."""
    found = re.search(pattern, (RESULTS / report).read_text(encoding='utf-8'), re.M)
    if not found:
        raise LookupError(f'analysis/results/{report} has no line that matches {pattern!r}')
    return form.format(*found.groups())


def show(row, recomputed, stored_value):
    """Print the row's line and keep it for the table at the end."""
    equal = 'n/a' if stored_value is None else 'yes' if recomputed == stored_value else 'no'
    stored_value = stored_value or 'no stored report yet'
    TABLE[row] = (recomputed, stored_value, equal)
    print(f'recomputed: {recomputed} | stored by the registered analysis: {stored_value} | equal: {equal}')


def ready(rows, *inputs):
    """True if every input of a row is on this machine. If one is not, say which: nothing is approximated."""
    missing = [p.relative_to(ROOT).as_posix() for p in inputs if not p.exists()]
    if missing:
        for row in ([rows] if isinstance(rows, str) else rows):
            TABLE[row] = (f'not recomputed: no {missing[0]}', 'not compared', 'n/a')
        print(f'not recomputed: this machine has no {" and no ".join(missing)} (see the list under the first cell)')
    return not missing


def in_repository(paths):
    """For each path, whether git tracks it (for a folder: a file in it); None when git cannot be asked."""
    command = ['git', '-C', str(ROOT), 'ls-files', '--', *paths]
    try:
        listed = subprocess.run(command, capture_output=True, text=True, check=True).stdout.split('\n')
    except (OSError, subprocess.CalledProcessError):
        return None
    return {path: any(name == path or name.startswith(path + '/') for name in listed) for path in paths}


RECORDS = [f'analysis/results/{run}/forks/{name}.jsonl' for run, name in (
    ('v5', 'results'), ('v6', 'results'), ('v8', 'results'), ('c', 'results'), ('qwen', 'results'),
    ('12', 'results'), ('13', 'results'), ('w', 'results'), ('k', 'results_k'), ('g', 'results_g'),
    ('09', 'results_09'))]
REPORTS = [f'analysis/results/{name}.txt' for name in (
    'analyze_a', 'analyze_b', 'explore_b', 'c/analyze_c', 'qwen/analyze_d2_qwen', '12/analyze_12',
    '13/analyze_13', 'w/analyze_w', 'k/analyze_k', 'g/analyze_g', '09/analyze_09', '09/analyze_11')]
INPUTS = (  # (the input and how it is made, its paths)
    (f'analysis/results/<run>/forks/results*.jsonl: the raw records of {len(RECORDS)} runs, as the runs wrote them',
     RECORDS),
    (f'analysis/results/**/*.txt: {len(REPORTS)} stored reports, as the analysis scripts printed them', REPORTS),
    ('analysis/fork_states/states.jsonl: paused steps, built by analysis/fork_states.py from public traces',
     ['analysis/fork_states/states.jsonl']),
    ('analysis/fork_states_c/states.jsonl: built by the same script, with that folder as its argument',
     ['analysis/fork_states_c/states.jsonl']),
    ('analysis/fork_states_d/states.jsonl: built by analysis/make_states_d.py from fork_states_c',
     ['analysis/fork_states_d/states.jsonl']),
    ('analysis/fork_states_k/population.json: built by analysis/make_population_k.py from fork_states_d',
     ['analysis/fork_states_k/population.json']),
    ('analysis/fork_states_g/fresh_states.jsonl: built by analysis/make_population_g.py from the same traces',
     ['analysis/fork_states_g/fresh_states.jsonl']),
    ('experiments/00-pointer-audit/results/units.csv: built by pointer_audit.py in that folder',
     ['experiments/00-pointer-audit/results/units.csv']),
    ('data/snapshots: competition data, not redistributed; download it as data/README.md says',
     ['data/snapshots']),
    ('data/cache/snapshot_files: built by analysis/snapshot_index.py from data/snapshots',
     ['data/cache/snapshot_files']),
    ('data/tasks.jsonl: competition data, read by the whole-run rows only', ['data/tasks.jsonl']),
    ('analysis/results/14/N and NAF: the whole runs of pre-registration 14',
     ['analysis/results/14/N', 'analysis/results/14/NAF']),
)
tracked = in_repository([path for _, paths in INPUTS for path in paths])
print('here: on this machine. repo: a file of the repository, as git reports it.')
print('here  repo  input')
for what, paths in INPUTS:
    here = 'yes' if all((ROOT / path).exists() for path in paths) else 'no'
    repo = '?' if tracked is None else 'yes' if all(tracked[path] for path in paths) else 'no'
    print(f'{here:<5} {repo:<5} {what}')

here: on this machine. repo: a file of the repository, as git reports it.
here  repo  input
yes   yes   analysis/results/<run>/forks/results*.jsonl: the raw records of 11 runs, as the runs wrote them
yes   yes   analysis/results/**/*.txt: 12 stored reports, as the analysis scripts printed them
yes   yes   analysis/fork_states/states.jsonl: paused steps, built by analysis/fork_states.py from public traces
yes   yes   analysis/fork_states_c/states.jsonl: built by the same script, with that folder as its argument
yes   yes   analysis/fork_states_d/states.jsonl: built by analysis/make_states_d.py from fork_states_c
yes   yes   analysis/fork_states_k/population.json: built by analysis/make_population_k.py from fork_states_d
yes   yes   analysis/fork_states_g/fresh_states.jsonl: built by analysis/make_population_g.py from the same traces
yes   yes   experiments/00-pointer-audit/results/units.csv: built by pointer_audit.py in that folder
yes   no    data/snapshots: competition data, not redis

## 02 H1: bare names against no list

**Rule, as registered** (`prereg/02-discrimination-forks.md`): "H1, following above base rate: the lower bound of
the CI for [P(any unopened true | P and H) minus P(any unopened listed true | A)] is above 0."

A is *no list*. P and H are *bare names*, delivered as the tool's own result (P) or as a note (H); the rule pools
them. The outcome is that the next call reads a neighbor's code. A paused step is analyzed when all five conditions
of the run (the other two carry a false entry) returned a well-formed call for all three seeds. The stored report
prints proportions to four decimals; both sides are shown in percentage points.

In [2]:
ROW = '02 H1'
if ready(ROW, RESULTS / 'v5/forks/results.jsonl', ANALYSIS / 'fork_states/states.jsonl', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states/states.jsonl')
    conditions = ('A', 'P', 'Pp', 'H', 'Hp')   # the five conditions of this run
    replays = usable(read_jsonl(RESULTS / 'v5/forks/results.jsonl'), conditions)
    steps = complete(replays, conditions)
    # the outcome of one replay: its next call reads a neighbor's code
    reads = lambda k, replay: any_true(paused[k], replay['next_call'])
    as_result, as_note = outcomes(steps, replays, 'P', reads), outcomes(steps, replays, 'H', reads)
    tables = {'no list': outcomes(steps, replays, 'A', reads),
              'bare names': {k: as_result[k] + as_note[k] for k in steps}}   # the rule pools the two deliveries
    rates(tables, f'{len(steps)} paused steps')
    estimate = diff_ci(tables['bare names'], tables['no list'])
    show(ROW, in_points(as_printed(estimate)),
         in_points(stored('analyze_a.txt', r'^  H1 diff: (\S+) \[(\S+), (\S+)\]')))

no list: 25 of 1179 (2.12%) | bare names: 69 of 2358 (2.93%) | 393 paused steps
recomputed: +0.81 [-0.08, +1.65] | stored by the registered analysis: +0.81 [-0.08, +1.65] | equal: yes


## 03 B3: addresses against bare names

**Rule** (`prereg/03-directive-and-encoding.md`): "B3, encoding (two-sided): P(any unopened true neighbor touched |
E) minus P(the same | H); the 95% interval excludes 0."

E is *addresses*: each neighbor with its file, line range and first definition line. H is *bare names*. The
registered analysis pools the three hosted runs (ten conditions) and analyzes the paused steps that are complete in
all ten. The stored report prints proportions; both sides are shown in percentage points.

In [3]:
ROW = '03 B3'
hosted = [RESULTS / f'{run}/forks/results.jsonl' for run in ('v5', 'v6', 'v8')]   # the three hosted runs
if ready(ROW, *hosted, ANALYSIS / 'fork_states/states.jsonl', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states/states.jsonl')
    conditions = ('A', 'P', 'Pp', 'H', 'Hp', 'D', 'Dp', 'E', 'RD', 'RE')   # the ten conditions of the three runs
    replays = usable([r for path in hosted for r in read_jsonl(path)], conditions)
    steps = complete(replays, conditions)
    # the outcome of one replay: its next call reads a neighbor's code
    reads = lambda k, replay: any_true(paused[k], replay['next_call'])
    tables = {'bare names': outcomes(steps, replays, 'H', reads),
              'addresses': outcomes(steps, replays, 'E', reads)}
    rates(tables, f'{len(steps)} paused steps')
    estimate = diff_ci(tables['addresses'], tables['bare names'])
    show(ROW, in_points(as_printed(estimate)),
         in_points(stored('analyze_b.txt', r'^  B3: (\S+) \[(\S+), (\S+)\]')))

bare names: 32 of 1164 (2.75%) | addresses: 96 of 1164 (8.25%) | 388 paused steps
recomputed: +5.50 [+3.44, +7.65] | stored by the registered analysis: +5.50 [+3.44, +7.65] | equal: yes


## 10 M2: the same on Qwen2.5-Coder-32B, a model of another family

**Rule** (`prereg/10-another-model-family.md`): "M2. E minus H on any unopened true neighbor touched (the outcome
of pre-registrations 02 to 05). 'The encoding effect replicates on a model of another family' if the lower 95% bound
is above 0; otherwise 'not detected on this model'."

The contrast of 03 B3, with Qwen2.5-Coder-32B continuing the same paused steps. The run has four conditions (no
list, bare names, addresses, rewired list); a paused step is analyzed when all four are usable.

In [4]:
ROW = '10 M2'
if ready(ROW, RESULTS / 'qwen/forks/results.jsonl', ANALYSIS / 'fork_states_d/states.jsonl', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    conditions = ('A', 'H', 'E', 'RE')   # no list, bare names, addresses, rewired list
    replays = usable(read_jsonl(RESULTS / 'qwen/forks/results.jsonl'), conditions)
    steps = complete(replays, conditions)
    # the outcome of one replay: its next call reads a neighbor's code
    reads = lambda k, replay: any_true(paused[k], replay['next_call'])
    tables = {'bare names': outcomes(steps, replays, 'H', reads),
              'addresses': outcomes(steps, replays, 'E', reads)}
    rates(tables, f'{len(steps)} paused steps')
    show(ROW, points(diff_ci(tables['addresses'], tables['bare names'])),
         stored('qwen/analyze_d2_qwen.txt', r'^  M1 E - H 95%: (\S+) \[(\S+), (\S+)\]'))

bare names: 29 of 891 (3.25%) | addresses: 66 of 891 (7.41%) | 297 paused steps
recomputed: +4.15 [+1.91, +6.62] | stored by the registered analysis: +4.15 [+1.91, +6.62] | equal: yes


## 12 P2: file only against bare names, opens a neighbor's file

**Rule** (`prereg/12-file-without-lines.md`): "P2. EPATH minus H on outcome 2. 'A path alone gets the agent to the
file' if the lower bound is above 0." Outcome 2: the next call is a `read_file` of a file that holds a listed
neighbor and that the agent had not opened, at any line.

EPATH is *file only*: each neighbor's name and file. H (*bare names*) and EL (*file and lines*) are not run again:
they come from the run of pre-registration 04, on the same platform, paused steps and seeds. A paused step is
analyzed when H, EL and EPATH are usable.

In [5]:
ROW = '12 P2'
if ready(ROW, RESULTS / '12/forks/results.jsonl', RESULTS / 'c/forks/results.jsonl',
         ANALYSIS / 'fork_states_d/states.jsonl'):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    reference = usable(read_jsonl(RESULTS / 'c/forks/results.jsonl'), ('H', 'EL'))   # the run of pre-registration 04
    new = usable(read_jsonl(RESULTS / '12/forks/results.jsonl'), ('EPATH',))         # the run of pre-registration 12
    replays = {k: {**reference.get(k, {}), **new.get(k, {})} for k in paused}
    steps = complete(replays, ('H', 'EL', 'EPATH'))
    # the outcome of one replay: its next call opens a neighbor's file
    opens = lambda k, replay: opens_listed_file(paused[k], replay['next_call'])
    tables = {'bare names': outcomes(steps, replays, 'H', opens),
              'file only': outcomes(steps, replays, 'EPATH', opens)}
    rates(tables, f'{len(steps)} paused steps')
    show(ROW, points(diff_ci(tables['file only'], tables['bare names'])),
         stored('12/analyze_12.txt', r'^  P2, .*?: (\S+) \[(\S+), (\S+)\]'))

bare names: 88 of 1206 (7.30%) | file only: 104 of 1206 (8.62%) | 402 paused steps
recomputed: +1.33 [-0.08, +2.82] | stored by the registered analysis: +1.33 [-0.08, +2.82] | equal: yes


## 12 P1: file and lines against file only, reads a neighbor's code

**Rule** (`prereg/12-file-without-lines.md`): "P1. EL minus EPATH on outcome 1. 'The line range, not the path, gets
the agent to the code' if the lower bound is above 0." Outcome 1: the next call reads a neighbor's code.

The same records and the same complete paused steps as 12 P2.

In [6]:
ROW = '12 P1'
if ready(ROW, RESULTS / '12/forks/results.jsonl', RESULTS / 'c/forks/results.jsonl',
         ANALYSIS / 'fork_states_d/states.jsonl', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    reference = usable(read_jsonl(RESULTS / 'c/forks/results.jsonl'), ('H', 'EL'))   # the run of pre-registration 04
    new = usable(read_jsonl(RESULTS / '12/forks/results.jsonl'), ('EPATH',))         # the run of pre-registration 12
    replays = {k: {**reference.get(k, {}), **new.get(k, {})} for k in paused}
    steps = complete(replays, ('H', 'EL', 'EPATH'))
    # the outcome of one replay: its next call reads a neighbor's code
    reads = lambda k, replay: any_true(paused[k], replay['next_call'])
    tables = {'file only': outcomes(steps, replays, 'EPATH', reads),
              'file and lines': outcomes(steps, replays, 'EL', reads)}
    rates(tables, f'{len(steps)} paused steps')
    show(ROW, points(diff_ci(tables['file and lines'], tables['file only'])),
         stored('12/analyze_12.txt', r'^  P1, .*?: (\S+) \[(\S+), (\S+)\]'))

file only: 38 of 1206 (3.15%) | file and lines: 93 of 1206 (7.71%) | 402 paused steps
recomputed: +4.56 [+2.74, +6.63] | stored by the registered analysis: +4.56 [+2.74, +6.63] | equal: yes


## 12 P3: with the file only, first reads that ask past line 1

**Rule** (`prereg/12-file-without-lines.md`): "P3. Under EPATH, at most 5% of the first reads of a listed neighbor's
file ask past line 1: 'without a line number the file is read from the top'. Tested only if there are at least 30
such reads."

The rule sets a threshold on a share, so the row is a count and has no interval. A first read here is a replay
whose next call is a `read_file` of a neighbor's file that the agent had not opened; it asks past line 1 when the
start line the model wrote is above 1.

In [7]:
ROW = '12 P3'
if ready(ROW, RESULTS / '12/forks/results.jsonl', RESULTS / 'c/forks/results.jsonl',
         ANALYSIS / 'fork_states_d/states.jsonl'):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    reference = usable(read_jsonl(RESULTS / 'c/forks/results.jsonl'), ('H', 'EL'))   # the run of pre-registration 04
    new = usable(read_jsonl(RESULTS / '12/forks/results.jsonl'), ('EPATH',))         # the run of pre-registration 12
    replays = {k: {**reference.get(k, {}), **new.get(k, {})} for k in paused}
    steps = complete(replays, ('H', 'EL', 'EPATH'))
    calls = [(k, replays[k][('EPATH', seed)]['next_call']) for k in steps for seed in SEEDS]
    first_reads = [(k, call) for k, call in calls
                   if call.get('function_name') == 'read_file' and opens_listed_file(paused[k], call)]
    # read_stats(...)[0]: the start line the model wrote is above 1
    past_line_1 = sum(read_stats(paused[k], call)[0] for k, call in first_reads)
    print(f'file only: {len(first_reads)} first reads of a neighbor\'s file in {len(calls)} replays')
    show(ROW, f'{past_line_1} of {len(first_reads)}',
         stored('12/analyze_12.txt', r'^  P3, .*?: (\d+) of (\d+)', '{} of {}'))

file only: 104 first reads of a neighbor's file in 1206 replays
recomputed: 0 of 104 | stored by the registered analysis: 0 of 104 | equal: yes


## 13 X1: the address printed as a search hit against the address

**Rule** (`prereg/13-address-as-search-hit.md`): "X1. EG minus E on outcome 1. 'Written as a search hit, the address
is followed more' if the lower bound is above 0; 'followed less' if the upper bound is below 0; otherwise 'no
detectable difference'." Outcome 1: the next call reads a neighbor's code.

EG prints each neighbor the way a numbered search prints a hit: `file:line:definition line`. E (*addresses*) comes
from the run of pre-registration 04. A paused step is analyzed when E and EG are usable.

In [8]:
ROW = '13 X1'
if ready(ROW, RESULTS / '13/forks/results.jsonl', RESULTS / 'c/forks/results.jsonl',
         ANALYSIS / 'fork_states_d/states.jsonl', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    reference = usable(read_jsonl(RESULTS / 'c/forks/results.jsonl'), ('E',))   # the run of pre-registration 04
    new = usable(read_jsonl(RESULTS / '13/forks/results.jsonl'), ('EG',))       # the run of pre-registration 13
    replays = {k: {**reference.get(k, {}), **new.get(k, {})} for k in paused}
    steps = complete(replays, ('E', 'EG'))
    # the outcome of one replay: its next call reads a neighbor's code
    reads = lambda k, replay: any_true(paused[k], replay['next_call'])
    tables = {'addresses': outcomes(steps, replays, 'E', reads),
              'as a search hit': outcomes(steps, replays, 'EG', reads)}
    rates(tables, f'{len(steps)} paused steps')
    show(ROW, points(diff_ci(tables['as a search hit'], tables['addresses'])),
         stored('13/analyze_13.txt', r'^  X1, .*?: (\S+) \[(\S+), (\S+)\]'))

addresses: 104 of 1206 (8.62%) | as a search hit: 91 of 1206 (7.55%) | 402 paused steps
recomputed: -1.08 [-2.74, +0.58] | stored by the registered analysis: -1.08 [-2.74, +0.58] | equal: yes


## 11 R1: fresh episodes, first reads that ask past line 1, after a line number against after a path only

**Rule** (`prereg/11-reading-rule-fresh-episodes.md`): "R1, shell output. All three arms together; the class of a
read depends only on the call just before. 'The reading rule holds in our fresh episodes' if all three are true:
(a) at least 65% of first reads after an ADDRESS ask past line 1; (b) at most 5% of first reads after a PATH ONLY
do; (c) the difference has a lower bound above 50 points."

This row is observational: nobody chose what the agent was shown. The records are the fresh episodes of
pre-registration 09 in its three conditions. A first read is an executed `read_file` of a file that the episode has
not read or viewed before. After an ADDRESS: the result of the call just before names that file together with a
line number, as a numbered search or listing prints it. After a PATH ONLY: it names the file and no line.

The row of Table 2 is the two counts. The difference of condition (c), with its interval over tasks, is printed
above the row's line; the cell stops with an error if it is not the one in the stored report.

In [9]:
ROW = '11 R1'
if ready(ROW, RESULTS / '09/forks/results_09.jsonl'):
    episodes = [r for r in read_jsonl(RESULTS / '09/forks/results_09.jsonl')
                if r.get('arm') in ('N', 'NA', 'NAF') and not r.get('error')]
    first_reads = [read for r in episodes for read in reads_of(r) if read['first']]
    tables = {'after a line number': collections.defaultdict(list),
              'after a path only': collections.defaultdict(list)}
    for read in first_reads:   # per task: 1 if the read asks past line 1, else 0
        if read['cls'] == 'ADDRESS':
            tables['after a line number'][read['task']].append(int(read['ranged']))
        elif read['cls'] == 'PATH ONLY':
            tables['after a path only'][read['task']].append(int(read['ranged']))
    rates(tables, f'{len(episodes)} episodes')
    in_report = (r'all arms \(N, NA, NAF\)\n\s+after an ADDRESS:\s+(\d+) of (\d+) .*\n'
                 r'\s+after a PATH ONLY:\s+(\d+) of (\d+) .*\n'
                 r'\s+ADDRESS minus PATH ONLY, by task: (\S+) points \[(\S+), (\S+)\]')
    difference = points(diff_ci(tables['after a line number'], tables['after a path only']), digits=1)
    assert difference == stored('09/analyze_11.txt', in_report, '{4} [{5}, {6}]'), 'not the stored difference'
    print(f'difference over tasks: {difference}, the same as in the stored report')
    show(ROW, f"{count(tables['after a line number'])} against {count(tables['after a path only'])}",
         stored('09/analyze_11.txt', in_report, '{0} of {1} against {2} of {3}'))

after a line number: 83 of 100 (83.00%) | after a path only: 0 of 708 (0.00%) | 945 episodes
difference over tasks: +83.0 [+66.0, +97.8], the same as in the stored report
recomputed: 83 of 100 against 0 of 708 | stored by the registered analysis: 83 of 100 against 0 of 708 | equal: yes


## 04 C1: true entry against false entry

**Rule** (`prereg/04-experiment-c-amendment.md`): "C1, the false edge under the legible encoding. D_E = P(swapped-out
entry touched | E) minus P(decoy touched | Ep). [...] 'Checks the edge': the lower 95% bound of D_E is above 0 and
the decoy rate is below half the swapped-out rate."

Under E (*addresses*) the list holds one chosen true neighbor. Under Ep (*false entry*) that neighbor is replaced by
a function that is not a neighbor, written with its own file, lines and definition line. The row of Table 2 counts
how often the next call reads the code of the true neighbor under E and of the false entry under Ep.

The difference D_E with its interval is printed above the row's line, in percentage points (the stored report
prints it as a proportion); the cell stops with an error if it is not the one in the stored report.

In [10]:
ROW = '04 C1'
if ready(ROW, RESULTS / 'c/forks/results.jsonl', ANALYSIS / 'fork_states_c/states.jsonl', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states_c/states.jsonl')
    conditions = ('H', 'E', 'Ep', 'EL', 'ES')   # the five conditions of this run
    replays = usable(read_jsonl(RESULTS / 'c/forks/results.jsonl'), conditions)
    steps = complete(replays, conditions)

    def reads_entry(k, replay, entry):
        """The next call reads the code of one entry: the chosen true neighbor, or the false entry in its place."""
        state = paused[k]
        name = state[entry]
        span = state['neighbor_spans'].get(name) if entry == 'swapped_out' else state['decoy_span']
        return touch_map(state, replay['next_call'], [name], {name: span}).get(name) in ('read', 'edit')

    tables = {'true entry': outcomes(steps, replays, 'E', lambda k, replay: reads_entry(k, replay, 'swapped_out')),
              'false entry': outcomes(steps, replays, 'Ep', lambda k, replay: reads_entry(k, replay, 'decoy'))}
    rates(tables, f'{len(steps)} paused steps')
    in_report = (r'^  D_E 95%: (\S+) \[(\S+), (\S+)\] \| '
                 r'swapped-out under E (\d+)/(\d+), decoy under Ep (\d+)/\5$')
    difference = in_points(as_printed(diff_ci(tables['true entry'], tables['false entry'])))
    assert difference == in_points(stored('c/analyze_c.txt', in_report)), 'not the stored difference'
    print(f'difference: {difference}, the same as in the stored report')
    hits = {name: sum(v for per_step in table.values() for v in per_step) for name, table in tables.items()}
    show(ROW, f"{hits['true entry']} against {hits['false entry']} of {len(steps) * len(SEEDS)}",
         stored('c/analyze_c.txt', in_report, '{3} against {5} of {4}'))

true entry: 23 of 1200 (1.92%) | false entry: 1 of 1200 (0.08%) | 400 paused steps
difference: +1.83 [+0.83, +3.17], the same as in the stored report
recomputed: 23 against 1 of 1200 | stored by the registered analysis: 23 against 1 of 1200 | equal: yes


## 03 B4: true list against rewired list, both under the registered filter

**Rule** (`prereg/03-directive-and-encoding.md`, amendment): "B4, structure versus vocabulary: at each dose (D versus
R_D, E versus R_E), the difference between any-true following under the true list and any-rewired following under
the rewired list. 'Structure matters' if, in at least one dose, the true list beats the rewired list by at least 3
points with a 95% clustered interval excluding 0."

The *rewired list* (R_E) holds the function's neighbors in a graph rewired with degrees kept, written with
addresses; the true list is E. The amendment says that "the rewired entries are, like the true entries, in unopened
files and unseen names". The rewired entries were built under that filter: a name the agent has not seen, in a file
it has not opened, outside the reference patch. The registered script scores every true neighbor in an unopened
file, without the filter (its line `E minus R_E` in `analysis/results/analyze_b.txt`). The row of Table 2 applies
the filter to both lists. Its stored value is the line `E matched true - RE rewired` of
`analysis/results/explore_b.txt`, written by `analysis/explore_b.py`: an exploratory script, which
`analysis/reproduce.py` does not list.

In [11]:
ROW = '03 B4'
hosted = [RESULTS / f'{run}/forks/results.jsonl' for run in ('v5', 'v6', 'v8')]   # the three hosted runs
units = ROOT / 'experiments/00-pointer-audit/results/units.csv'                  # what each reference patch changes
if ready(ROW, *hosted, ANALYSIS / 'fork_states/states.jsonl', units, SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states/states.jsonl')
    conditions = ('A', 'P', 'Pp', 'H', 'Hp', 'D', 'Dp', 'E', 'RD', 'RE')   # the ten conditions of the three runs
    replays = usable([r for path in hosted for r in read_jsonl(path)], conditions)
    steps = complete(replays, conditions)
    patch_ids, patch_files = load_gold()   # per task: the functions and the files that the reference patch changes

    def filtered(state):
        """The true neighbors that pass the filter the rewired entries were built with."""
        unopened, unseen = set(state['unshown_true_neighbors']), set(state['fresh_true_neighbors'])
        in_patch = set(state['gold_ids']) | patch_ids[state['task']]
        spans = state['neighbor_spans']
        return [n for n in state['true_neighbors'] if spans.get(n) and n in unopened and n in unseen
                and n not in in_patch and spans[n]['path'] not in patch_files[state['task']]]

    def reads_any(k, replay, names, spans):
        """The next call reads the code of one of these entries."""
        touched = touch_map(paused[k], replay['next_call'], names, spans)
        return any(how in ('read', 'edit') for how in touched.values())

    tables = {'rewired list': outcomes(steps, replays, 'RE', lambda k, replay: reads_any(
                  k, replay, paused[k]['rewired_entries'], paused[k]['rewired_spans'])),
              'true list': outcomes(steps, replays, 'E', lambda k, replay: reads_any(
                  k, replay, filtered(paused[k]), paused[k]['neighbor_spans']))}
    rates(tables, f'{len(steps)} paused steps')
    show(ROW, points(diff_ci(tables['true list'], tables['rewired list'])),
         stored('explore_b.txt', r'^  E matched true - RE rewired, touch\s+(\S+) \[ *(\S+), +(\S+)\]'))

rewired list: 12 of 1164 (1.03%) | true list: 29 of 1164 (2.49%) | 388 paused steps
recomputed: +1.46 [+0.00, +3.01] | stored by the registered analysis: +1.46 [+0.00, +3.01] | equal: yes


## 06 K1: five calls, addresses against bare names, shown the fix

**Rule** (`prereg/06-navigation-horizon.md`): "K1, reaching the fix. E minus H on Yline, in R. 'The legible list
brings the agent to fix code within five calls' if the lower 95% bound is above 0; otherwise 'no detectable gain'."

Five-call replays: the agent goes on for up to five model calls, and reads and read-only shell commands execute. R
is the set of paused steps where a listed neighbor lies in a file that the reference patch changes (five seeds).
Yline is *shown the fix*: within the five calls a tool result shows a line that the reference patch changes. A
replay is usable when it ended without an error and its first response holds a well-formed call.

In [12]:
ROW = '06 K1'
if ready(ROW, RESULTS / 'k/forks/results_k.jsonl', ANALYSIS / 'fork_states_d/states.jsonl',
         ANALYSIS / 'fork_states_k/population.json', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    population = read_json(ANALYSIS / 'fork_states_k/population.json')
    fix = {tuple(entry['key']): entry for entry in population['R']}   # R, with what the reference patch changes
    seeds = population['seeds']['R']
    conditions = ('A', 'H', 'E', 'RE')   # no list, bare names, addresses, rewired list
    records = [r for r in read_jsonl(RESULTS / 'k/forks/results_k.jsonl') if step_of(r) in fix]
    replays = usable(records, conditions, seeds, broken=first_call_failure)
    steps = complete(replays, conditions, seeds)
    # the outcome of one replay: it is shown the fix within its five calls
    shown = lambda k, replay: score(paused[k], fix[k], replay)['Yline']
    tables = {'bare names': outcomes(steps, replays, 'H', shown, seeds),
              'addresses': outcomes(steps, replays, 'E', shown, seeds)}
    rates(tables, f'{len(steps)} paused steps, {len(seeds)} seeds')
    show(ROW, points(diff_ci(tables['addresses'], tables['bare names'])),
         stored('k/analyze_k.txt', r'^  Yline E - H: (\S+) \[(\S+), (\S+)\]  \(K1\)'))

bare names: 188 of 285 (65.96%) | addresses: 199 of 285 (69.82%) | 57 paused steps, 5 seeds
recomputed: +3.86 [-1.05, +8.77] | stored by the registered analysis: +3.86 [-1.05, +8.77] | equal: yes


## 09 A1: fresh episodes, the next call reads a returned entity, addressed against as shipped

**Rule** (`prereg/09-fresh-episodes-addressed-graph-tools.md`): "A1, addresses in the tool's own answer. NA minus N
on Ynext, over all episodes of complete tasks. 'Addresses in the graph tools' own answers bring the next call to a
returned entity's code' if the lower 95% bound is above 0; otherwise 'no detectable difference'."

Fresh episodes: the agent starts from the task prompt and makes up to eight model calls. N is *as shipped*; NA is
*addressed*. Ynext is 1 when a graph answer of the episode returns an entity with a line range in a file not yet
read (the first such answer counts) and the first call of the next response reads that entity's code; an episode
without such an answer scores 0. A task is analyzed when its nine episodes (three conditions, three seeds) finished
without an error. The interval is over tasks.

In [13]:
ROW = '09 A1'
if ready(ROW, RESULTS / '09/forks/results_09.jsonl', ANALYSIS / 'fork_states_g/fresh_states.jsonl',
         SNAPSHOTS, INDEX):
    tasks = {state['task'] for state in read_jsonl(ANALYSIS / 'fork_states_g/fresh_states.jsonl')}
    conditions = ('N', 'NA', 'NAF')   # as shipped, addressed, addressed and repaired
    records = [r for r in read_jsonl(RESULTS / '09/forks/results_09.jsonl')
               if r.get('population') == 'FRESH' and r['task'] in tasks]
    episodes = usable(records, conditions, broken=lambda r: r.get('error'), cluster=lambda r: r['task'])
    done = complete(episodes, conditions)   # tasks whose nine episodes finished without an error
    # the outcome of one episode: the call after the graph answer reads an entity that the answer returned
    follows = lambda task, r: episode(task, r['arm'], r)['Ynext']
    tables = {'as shipped': outcomes(done, episodes, 'N', follows),
              'addressed': outcomes(done, episodes, 'NA', follows)}
    rates(tables, f'{len(done)} tasks')
    show(ROW, points(diff_ci(tables['addressed'], tables['as shipped'])),
         stored('09/analyze_09.txt',
                r'^  A1 NA - N on Ynext, clustered by task \(\d+ tasks\): (\S+) \[(\S+), (\S+)\]'))

as shipped: 12 of 297 (4.04%) | addressed: 23 of 297 (7.74%) | 99 tasks
recomputed: +3.70 [+1.35, +6.73] | stored by the registered analysis: +3.70 [+1.35, +6.73] | equal: yes


## 07 F2: misspelled line bound, dictionary rendering against the earlier rendering

**Rule** (`prereg/07-tool-result-rendering.md`): "F2, mangled names. Share of usable replays whose first call is a
`read_file` with a mangled line-bound name (a name that is not a plain identifier and reads `start_line` or
`end_line` once quotes, backslashes and spaces are removed), arm A minus arm Aw. 'The v28 rendering raises mangled
bound names' if the lower 95% bound is above 0; otherwise no detectable difference."

A replays the paused step with the earlier tool results rendered as dictionaries, as the harness has delivered them
since Sep 30. Aw renders them as before, as a JSON string. Neither adds a list. A paused step is analyzed when the
four conditions of the run are usable.

In [14]:
ROW = '07 F2'
if ready(ROW, RESULTS / 'w/forks/results.jsonl'):
    conditions = ('A', 'Aw', 'Hw', 'Ew')   # the four conditions of this run
    replays = usable(read_jsonl(RESULTS / 'w/forks/results.jsonl'), conditions)
    steps = complete(replays, conditions)
    # the outcome of one replay: its next call carries a line bound under a misspelled argument name
    misspelled = lambda k, replay: mangled(replay['next_call'])
    tables = {'earlier rendering': outcomes(steps, replays, 'Aw', misspelled),
              'dictionary rendering': outcomes(steps, replays, 'A', misspelled)}
    rates(tables, f'{len(steps)} paused steps')
    show(ROW, points(diff_ci(tables['dictionary rendering'], tables['earlier rendering'])),
         stored('w/analyze_w.txt', r'^  F2 A - Aw: (\S+) \[(\S+), (\S+)\]'))

earlier rendering: 2 of 1200 (0.17%) | dictionary rendering: 220 of 1200 (18.33%) | 400 paused steps
recomputed: +18.17 [+14.75, +21.75] | stored by the registered analysis: +18.17 [+14.75, +21.75] | equal: yes


## 08 N1: misspelled bounds among reads with a line range, fresh episodes as shipped

**Rule** (`prereg/08-fresh-episodes-and-patched-harness.md`): "N1, the defect in fresh episodes. Among executed
`read_file` calls that carry a line bound [...], the share whose bound name is mangled (not a plain identifier). 95%
bootstrap interval clustered by task. 'The defect occurs in fresh episodes' if the lower bound is above 2.1%, the
share in the 318 original traces (42 of 2,012); otherwise 'not shown above the original traces' share'."

An amendment of that pre-registration places these episodes inside the run of pre-registration 09: the rule is
evaluated on its *as shipped* episodes that finished without an error. The row is a share with its interval, not a
difference.

In [15]:
ROW = '08 N1'
if ready(ROW, RESULTS / '09/forks/results_09.jsonl', ANALYSIS / 'fork_states_g/fresh_states.jsonl'):
    tasks = {state['task'] for state in read_jsonl(ANALYSIS / 'fork_states_g/fresh_states.jsonl')}
    misspelled = collections.defaultdict(list)   # per task: 0 or 1 for each executed read that carries a line bound
    for r in read_jsonl(RESULTS / '09/forks/results_09.jsonl'):
        if (r.get('population') == 'FRESH' and r['arm'] == 'N' and r['seed'] in SEEDS and r['task'] in tasks
                and not r.get('error')):
            for _model_call, bound_is_misspelled, _first_line_returned in ranged_reads(r):
                misspelled[r['task']].append(bound_is_misspelled)
    rates({'reads with a line range whose bound is misspelled': misspelled}, f'{len(misspelled)} tasks')
    share = boot_ci(misspelled)
    show(ROW, '{:.2f}% [{:.2f}, {:.2f}]'.format(*(100 * x for x in share)),
         stored('09/analyze_09.txt', r'^  N1 share with a mangled bound name: (\S+) \[(\S+), (\S+)\]',
                '{}% [{}, {}]'))

reads with a line range whose bound is misspelled: 66 of 501 (13.17%) | 88 tasks
recomputed: 13.17% [9.19, 17.17] | stored by the registered analysis: 13.17% [9.19, 17.17] | equal: yes


## 08 G2: five calls, repair against as shipped, shown the fix

**Rule** (`prereg/08-fresh-episodes-and-patched-harness.md`): "G2, the argument fix alone. AF minus A on Yline, same
states and seeds. 'Honoring the bound alone raises reach within five calls' if the lower 95% bound is above 0;
otherwise 'no detectable gain from the argument fix alone'." The states and seeds are those of the rule before it:
all complete states of R and U together, seeds 11 to 13.

Five-call replays on the current harness. A is *as shipped*, with no list. AF is the *repair*: a misspelled
line-bound name is restored before the read runs. R as in 06 K1; U is the set of paused steps where a file that the
reference patch changes had not been opened and no listed neighbor lies in one. A paused step is complete when the
three conditions are usable for every seed that ran (five in R, three in U).

In [16]:
ROW = '08 G2'
if ready(ROW, RESULTS / 'g/forks/results_g.jsonl', ANALYSIS / 'fork_states_d/states.jsonl',
         ANALYSIS / 'fork_states_k/population.json', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    population = read_json(ANALYSIS / 'fork_states_k/population.json')
    ran = {'R': (11, 12, 13, 14, 15), 'U': (11, 12, 13)}   # the seeds that ran in each set of paused steps
    fix = {tuple(entry['key']): entry for name in ran for entry in population[name]}
    seeds_of = {tuple(entry['key']): ran[name] for name in ran for entry in population[name]}
    conditions = ('A', 'AF', 'EF')   # as shipped; repaired; repaired, with addresses
    records = [r for r in read_jsonl(RESULTS / 'g/forks/results_g.jsonl')
               if r.get('population') in ran and step_of(r) in fix and r['seed'] in seeds_of[step_of(r)]]
    replays = usable(records, conditions, ran['R'], broken=first_call_failure)
    steps = [k for k in sorted(replays)
             if {(condition, seed) for condition in conditions for seed in seeds_of[k]} <= set(replays[k])]
    # the outcome of one replay: it is shown the fix within its five calls
    shown = lambda k, replay: score(paused[k], fix[k], replay)['Yline']
    seeds = (11, 12, 13)   # the seeds that every paused step has
    tables = {'as shipped': outcomes(steps, replays, 'A', shown, seeds),
              'repaired': outcomes(steps, replays, 'AF', shown, seeds)}
    rates(tables, f'{len(steps)} paused steps')
    show(ROW, points(diff_ci(tables['repaired'], tables['as shipped'])),
         stored('g/analyze_g.txt',
                r'^  G2 AF - A on Yline, clustered by state \(\d+ clusters\): (\S+) \[(\S+), (\S+)\]'))

as shipped: 144 of 510 (28.24%) | repaired: 172 of 510 (33.73%) | 170 paused steps
recomputed: +5.49 [+2.35, +8.82] | stored by the registered analysis: +5.49 [+2.35, +8.82] | equal: yes


## 08 G3: five calls, addresses on top of the repair

**Rule** (`prereg/08-fresh-episodes-and-patched-harness.md`): "G3, the list beyond the argument fix. EF minus AF on
Yline in R, seeds 11 to 15. 'The legible list adds reach beyond the argument fix' if the lower 95% bound is above
0; otherwise 'no detectable gain from the list once the bound is honored'."

EF is the repair with the list of *addresses* added. The same run and the same complete paused steps as 08 G2,
restricted to R, with all five seeds.

In [17]:
ROW = '08 G3'
if ready(ROW, RESULTS / 'g/forks/results_g.jsonl', ANALYSIS / 'fork_states_d/states.jsonl',
         ANALYSIS / 'fork_states_k/population.json', SNAPSHOTS):
    paused = load_paused_steps(ANALYSIS / 'fork_states_d/states.jsonl')
    population = read_json(ANALYSIS / 'fork_states_k/population.json')
    fix = {tuple(entry['key']): entry for entry in population['R']}   # R, with what the reference patch changes
    seeds = (11, 12, 13, 14, 15)     # the seeds that ran in R
    conditions = ('A', 'AF', 'EF')   # as shipped; repaired; repaired, with addresses
    records = [r for r in read_jsonl(RESULTS / 'g/forks/results_g.jsonl')
               if r.get('population') == 'R' and step_of(r) in fix]
    replays = usable(records, conditions, seeds, broken=first_call_failure)
    steps = complete(replays, conditions, seeds)
    # the outcome of one replay: it is shown the fix within its five calls
    shown = lambda k, replay: score(paused[k], fix[k], replay)['Yline']
    tables = {'repaired': outcomes(steps, replays, 'AF', shown, seeds),
              'repaired, with addresses': outcomes(steps, replays, 'EF', shown, seeds)}
    rates(tables, f'{len(steps)} paused steps, {len(seeds)} seeds')
    show(ROW, points(diff_ci(tables['repaired, with addresses'], tables['repaired'])),
         stored('g/analyze_g.txt', r'^  G3 EF - AF on Yline in R, seeds 11 to 15, clustered by state '
                                   r'\(\d+ clusters\): (\S+) \[(\S+), (\S+)\]'))

repaired: 213 of 290 (73.45%) | repaired, with addresses: 213 of 290 (73.45%) | 58 paused steps, 5 seeds
recomputed: +0.00 [-5.86, +5.52] | stored by the registered analysis: +0.00 [-5.86, +5.52] | equal: yes


## 09 A2: fresh episodes, both changes against as shipped, shown the fix within eight calls

**Rule** (`prereg/09-fresh-episodes-addressed-graph-tools.md`): "A2, the patched harness in fresh episodes. NAF minus
N on Yline, over the complete tasks whose gold patch modifies a file. 'The patched harness brings the agent to the
fix's lines within eight calls of a fresh start' if the lower 95% bound is above 0; otherwise 'no detectable gain'."

NAF is *addressed and repaired*; N is *as shipped*. Yline is *shown the fix*: within the eight calls a tool result
shows a line that the reference patch changes. The interval is over tasks.

In [18]:
ROW = '09 A2'
if ready(ROW, RESULTS / '09/forks/results_09.jsonl', ANALYSIS / 'fork_states_g/fresh_states.jsonl', SNAPSHOTS):
    fresh = {state['task']: state for state in read_jsonl(ANALYSIS / 'fork_states_g/fresh_states.jsonl')}
    conditions = ('N', 'NA', 'NAF')   # as shipped, addressed, addressed and repaired
    records = [r for r in read_jsonl(RESULTS / '09/forks/results_09.jsonl')
               if r.get('population') == 'FRESH' and r['task'] in fresh]
    episodes = usable(records, conditions, broken=lambda r: r.get('error'), cluster=lambda r: r['task'])
    # complete tasks whose reference patch modifies a file
    done = [task for task in complete(episodes, conditions) if fresh[task]['gold_files']]

    def shown(task, r):
        """The outcome of one episode: within its eight calls a tool result shows a line that the fix changes."""
        changed = fresh[task]['gold_files']
        fix = {'gold_files': changed, 'unopened_gold_files': sorted(changed)}
        return score({'task': task, 'true_neighbors': [], 'neighbor_spans': {}}, fix, r)['Yline']

    tables = {'as shipped': outcomes(done, episodes, 'N', shown),
              'addressed and repaired': outcomes(done, episodes, 'NAF', shown)}
    rates(tables, f'{len(done)} tasks')
    show(ROW, points(diff_ci(tables['addressed and repaired'], tables['as shipped'])),
         stored('09/analyze_09.txt', r'^\s+A2 NAF - N on Yline, clustered by task: (\S+) \[(\S+), (\S+)\]'))

as shipped: 193 of 294 (65.65%) | addressed and repaired: 205 of 294 (69.73%) | 98 tasks
recomputed: +4.08 [+0.68, +7.48] | stored by the registered analysis: +4.08 [+0.68, +7.48] | equal: yes


## 14 T1, T2, T3: whole runs scored by tests

**Rules** (`prereg/14-whole-runs-scored-by-tests.md`): "Each is 'addressed and repaired minus as shipped', paired by
task, with a 95% bootstrap interval over tasks (2,000 resamples, seed 0). They apply if at least 60 tasks enter the
paired analysis." T1: resolved, the harness's verdict from the task's tests. T2: shown a line that the fix changes
within eight tool calls. T3: the final patch overlaps a changed region of the reference patch.

Whole runs: the agent works from the task prompt until it submits a patch or a budget ends, and the task's tests
score the patch. If the two result folders, `analysis/results/14/N` (as shipped) and `analysis/results/14/NAF`
(addressed and repaired), are not on this machine, the cell prints "pending". Otherwise it computes the three rows
as `analysis/analyze_14.py` defines them, with that script's reading and scoring of one run (`load_root`,
`score_run`): a task enters when both conditions have a result and neither failed for a reason outside the agent.
The stored values are read from `analysis/results/14/analyze_14.txt`.

In [19]:
ROWS = ('14 T1', '14 T2', '14 T3')
whole_runs = RESULTS / '14'
if not ((whole_runs / 'N').is_dir() and (whole_runs / 'NAF').is_dir()):
    for ROW in ROWS:
        TABLE[ROW] = ('pending', 'pending', 'n/a')
    print('pending: analysis/results/14/N and analysis/results/14/NAF are not here yet')
elif ready(ROWS, ROOT / 'data/tasks.jsonl', SNAPSHOTS):
    from analyze_14 import T2_K, load_root, score_run   # one whole run (trace, test result, patch) to its outcomes
    from public_runs_time_to_fix import gold_regions    # the regions that each reference patch changes
    fix = gold_regions()
    runs, _notes = load_root(whole_runs)   # {(task, condition, seed): (trace, test result, patch, extra)}
    scored = {key: score_run(key[0], trace, result, patch, fix)
              for key, (trace, result, patch, _extra) in runs.items()}
    pairs = sorted((task, seed) for task, condition, seed in scored
                   if condition == 'N' and (task, 'NAF', seed) in scored)
    # a pair enters when neither run failed for a reason outside the agent
    entered = [(task, seed) for task, seed in pairs
               if not scored[(task, 'N', seed)]['infra'] and not scored[(task, 'NAF', seed)]['infra']]
    # the line-level outcomes need a reference patch that modifies a file
    with_fix = [(task, seed) for task, seed in entered if scored[(task, 'N', seed)]['gold']]
    rules = (('14 T1', 'resolved', 'T1 resolved', entered, lambda run: run['resolved']),
             ('14 T2', f'shown the fix within {T2_K} calls', f'T2 t_shown <= {T2_K}', with_fix,
              lambda run: run['t'] <= T2_K),
             ('14 T3', 'patch on a changed region', 'T3 patch_overlaps', with_fix, lambda run: run['patch_region']))
    report = whole_runs / 'analyze_14.txt'
    for ROW, what, label, used, outcome in rules:
        tables = {'as shipped': collections.defaultdict(list),
                  'addressed and repaired': collections.defaultdict(list)}
        for task, seed in used:
            tables['as shipped'][task].append(int(outcome(scored[(task, 'N', seed)])))
            tables['addressed and repaired'][task].append(int(outcome(scored[(task, 'NAF', seed)])))
        rates(tables, f'{what}, {len(tables["as shipped"])} tasks')
        in_report = rf'^   {label}: (\S+) points \[(\S+), (\S+)\]'
        show(ROW, points(diff_ci(tables['addressed and repaired'], tables['as shipped']), digits=1),
             stored('14/analyze_14.txt', in_report) if report.is_file() else None)

as shipped: 36 of 103 (34.95%) | addressed and repaired: 37 of 103 (35.92%) | resolved, 103 tasks
recomputed: +1.0 [-5.8, +7.8] | stored by the registered analysis: +1.0 [-5.8, +7.8] | equal: yes
as shipped: 76 of 102 (74.51%) | addressed and repaired: 76 of 102 (74.51%) | shown the fix within 8 calls, 102 tasks
recomputed: +0.0 [-5.9, +4.9] | stored by the registered analysis: +0.0 [-5.9, +4.9] | equal: yes
as shipped: 58 of 102 (56.86%) | addressed and repaired: 60 of 102 (58.82%) | patch on a changed region, 102 tasks
recomputed: +2.0 [-4.9, +8.8] | stored by the registered analysis: +2.0 [-4.9, +8.8] | equal: yes


## The whole table

Every row again, as the cells above recorded it, and the count of rows whose recomputed value equals the stored one.

In [20]:
head = ('row', 'recomputed', 'stored by the registered analysis', 'equal')
lines = [head] + [(row, *values) for row, values in TABLE.items()]
widths = [max(len(line[i]) for line in lines) for i in range(4)]
for line in lines:
    print('  '.join(f'{text:<{width}}' for text, width in zip(line, widths)).rstrip())
compared = [equal for _, _, equal in TABLE.values() if equal != 'n/a']
print(f'\n{compared.count("yes")} of {len(compared)} recomputed rows equal the stored value; '
      f'{len(TABLE) - len(compared)} of {len(TABLE)} rows not compared (pending, or an input is missing)')

row    recomputed                  stored by the registered analysis  equal
02 H1  +0.81 [-0.08, +1.65]        +0.81 [-0.08, +1.65]               yes
03 B3  +5.50 [+3.44, +7.65]        +5.50 [+3.44, +7.65]               yes
10 M2  +4.15 [+1.91, +6.62]        +4.15 [+1.91, +6.62]               yes
12 P2  +1.33 [-0.08, +2.82]        +1.33 [-0.08, +2.82]               yes
12 P1  +4.56 [+2.74, +6.63]        +4.56 [+2.74, +6.63]               yes
12 P3  0 of 104                    0 of 104                           yes
13 X1  -1.08 [-2.74, +0.58]        -1.08 [-2.74, +0.58]               yes
11 R1  83 of 100 against 0 of 708  83 of 100 against 0 of 708         yes
04 C1  23 against 1 of 1200        23 against 1 of 1200               yes
03 B4  +1.46 [+0.00, +3.01]        +1.46 [+0.00, +3.01]               yes
06 K1  +3.86 [-1.05, +8.77]        +3.86 [-1.05, +8.77]               yes
09 A1  +3.70 [+1.35, +6.73]        +3.70 [+1.35, +6.73]               yes
07 F2  +18.17 [+14.75, +21.75]     +